Task 2 notes:
We're changing nn.binary_cross_entropy into nn.sigmoid_cross_entropy_with_logits.
For task 1 D(Z) gives out a probablity instead of the raw logits and the loss is then the BCE on this. We need to make D(x) give out raw logits. We decided to do this by removing "nn.sigmoid(" from D(x). G(z) still needs to output "real" pixel values so it keeps the "nn.sigmoid(" transformation since it always goes int D(x) before the loss function is used. 

In [1]:
#task 1 GPU accelerated
from comet_ml import Experiment
import torch
import torch.nn.functional as nn
import torch.autograd as autograd
import torch.optim as optim
import numpy as np
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os

from datetime import datetime




#GPU ACCELERATION
print("start")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
#GPU ACCELERATION


mb_size = 64 #mini batch size should be able raise to around 1024 for better performance


mnist = datasets.MNIST( #had to be replaced from the old tensorflow implementation
    root='./MNIST_data',
    train=True,
    download=True,
    transform=transforms.ToTensor()
)
#dataloader
data_loader = torch.utils.data.DataLoader(mnist, batch_size=mb_size, shuffle=True)
print(mnist)
X_dim = mnist[0][0].numel() #should be 28 * 28 = 784
y_dim = len(mnist.classes) #should be 10 (0->9)
Z_dim = 100 #random seed? size of latent space vector z that is input to the image generator
h_dim = 128 #hidden layer width
lr = 1e-3 #learning rate gamma 
ones_label = torch.ones(mb_size, 1)   
zeros_label = torch.zeros(mb_size, 1) #batch size big vector of 0



def xavier_init(size): #xavier intialisation of weights for the network. Variance = (1/I_n + O_n)
    in_dim = size[0] #neurons of input layer
    xavier_stddev = 1. / np.sqrt(in_dim / 2.) #actual creation of the weights
    return torch.randn(size) * xavier_stddev #changed to not use "variable()"
#Generator non gpu accelerated
#Wzh = torch.nn.Parameter(xavier_init(size=[Z_dim, h_dim])) #input(z) -> hidden layer weigth intialisation (100 * 128)
#bzh = torch.nn.Parameter(torch.zeros(h_dim)) #changed to not use "variable()" #bias = 0
#Whx = torch.nn.Parameter(xavier_init(size=[h_dim, X_dim])) #hidden -> image(x) weigth intialisation (128 * 784)
#bhx = torch.nn.Parameter(torch.zeros(X_dim))#changed to not use "variable()"#bias = 0

#Generator GPU accelerated
Wzh = torch.nn.Parameter(xavier_init(size=[Z_dim, h_dim]).to(device))
bzh = torch.nn.Parameter(torch.zeros(h_dim).to(device))
Whx = torch.nn.Parameter(xavier_init(size=[h_dim, X_dim]).to(device))
bhx = torch.nn.Parameter(torch.zeros(X_dim).to(device))

def G(z): #generator forward pass
    h = nn.relu(z @ Wzh + bzh.repeat(z.size(0), 1)) #input(z) -> hidden layer pass
    X = nn.sigmoid(h @ Whx + bhx.repeat(h.size(0), 1)) #hidden layer -> image pass
    return X #image

def reset_grad():
    for p in params:
        if p.grad is not None:#added this to avoid crashing. p.grad can be None
            p.grad.zero_()

#Discriminator non gpu accelerated
#Wxh = torch.nn.Parameter(xavier_init(size=[X_dim, h_dim])) #image(x) -> hidden layer weight initialisation (748 * 128)
#bxh = torch.nn.Parameter(torch.zeros(h_dim)) #bias = 0
#Why = torch.nn.Parameter(xavier_init(size=[h_dim, 1])) #hidden layer -> output(y) layer weight initialisation(128 * 1)
#bhy = torch.nn.Parameter(torch.zeros(1))

#Discriminator GPU accelerated
Wxh = torch.nn.Parameter(xavier_init(size=[X_dim, h_dim]).to(device))
bxh = torch.nn.Parameter(torch.zeros(h_dim).to(device))
Why = torch.nn.Parameter(xavier_init(size=[h_dim, 1]).to(device))
bhy = torch.nn.Parameter(torch.zeros(1).to(device))

def D(X): #discriminator forward pass
    h = nn.relu(X @ Wxh + bxh.repeat(X.size(0), 1)) #image(x) -> hidden layer pass

    #!!!!Task2
    y = h @ Why + bhy.repeat(h.size(0), 1)#image(x) -> hidden layer pass (real Logits)
    #!!!!Task2
    return y #output(real=1/fake=0)


G_params = [Wzh, bzh, Whx, bhx]#define generator parameters (pyTorch needs it like this)
D_params = [Wxh, bxh, Why, bhy] #define discriminator parameters (pyTorch needs it like this)
 
params = G_params + D_params

G_solver = optim.Adam(G_params, lr=1e-3) #Adaptive Moment Estimation(different learning rate per parameter + momentum)
D_solver = optim.Adam(D_params, lr=1e-3) #Adaptive Moment Estimation(different learning rate per parameter + momentum)


start
Using device: cuda
Dataset MNIST
    Number of datapoints: 60000
    Root location: ./MNIST_data
    Split: Train
    StandardTransform
Transform: ToTensor()


In [ ]:
#Task 2 training GPU accelerated:
run_name = datetime.now().strftime("Task2-%Y%m%d-%H%M%S")
#training
epochs = 500
experiment = Experiment(
    api_key="lZ6VM5qo8Rz8JKYXjXVS3zWzs",
    project_name="D7047E/Lab2",
    experiment_name=run_name  #
)
experiment.set_name(run_name)
experiment.log_parameters({
    "batch_size": mb_size,
    "Z_dim": Z_dim,
    "h_dim": h_dim,
    "learning_rate": lr,
    "epochs": epochs,
    "optimizer": "Adam",
    "loss_function": "nn.binary_cross_entropy_with_logits"
})

for epoch in range(epochs):
    print(f"Epoch {epoch}")
    d_loss_epoch_sum = 0
    d_loss_epoch_avg = 0
    d_loss_real_epoch_sum = 0
    d_loss_fake_epoch_sum = 0
    d_loss_real_epoch_avg = 0
    d_loss_fake_epoch_avg = 0
    
    g_loss_epoch_sum = 0
    g_loss_epoch_avg = 0
    
    data_iter = iter(data_loader) #redo for each epoch otherwise it goes through the same batch
    for it in range(60000//mb_size): # Sample data
        
        #D_solver.zero_grad()#reset gradient accumulation (not in article) but is more modern i think
        
        #z = torch.randn(mb_size, Z_dim) #random z vector of dimension batch size * z_dim (64*100)
        z = torch.randn(mb_size, Z_dim).to(device) #random z vector of dimension batch size * z_dim (64*100) GPU accelerated
        
        X, _ = next(data_iter) #get next sample from mnist
        #X = X.view(mb_size, -1) #flatten from [64, 1, 28, 28](mnist) to [batch size,784]
        X = X.view(mb_size, -1).to(device) #flatten from [64, 1, 28, 28](mnist) to [batch size,784] GPU accelerated
    
        #Discriminator forward-loss-backward-update
        G_sample = G(z)
        D_real = D(X)
        D_fake = D(G_sample)
        ones_label = torch.ones_like(D_real).to(device) #batch size big vector of 1 or smaller if D(X) gives out smaller batch
        zeros_label = torch.zeros_like(D_fake).to(device)#batch size big vector of 0 or smaller if D(X) gives out smaller batch
        #!!!!Task2
        D_loss_real = nn.binary_cross_entropy_with_logits(D_real, ones_label)
        D_loss_fake = nn.binary_cross_entropy_with_logits(D_fake, zeros_label)
        #!!!!Task2
        D_loss = D_loss_real + D_loss_fake
        d_loss_epoch_sum += D_loss.item()
        d_loss_real_epoch_sum += D_loss_real.item()
        d_loss_fake_epoch_sum += D_loss_fake.item()
        
        
        D_loss.backward()
        D_solver.step()
        reset_grad()
        # Generator forward-loss-backward-update
        z = torch.randn(mb_size, Z_dim).to(device) #random z vector of dimension batch size * z_dim (64*100) GPU accelerated
        G_sample = G(z)
        D_fake = D(G_sample)
        #!!!!Task2
        G_loss = nn.binary_cross_entropy_with_logits(D_fake, ones_label)
        #!!!!Task2
        g_loss_epoch_sum += G_loss.item() 
        G_loss.backward()
        G_solver.step()
        
        # Housekeeping - reset gradient
        reset_grad()
        
        if it % 100 == 0:
            print(f"Iter {it} | D_loss: {D_loss.item():.4f} | G_loss: {G_loss.item():.4f}")

    #log avg loss per epoch
    g_loss_epoch_avg = g_loss_epoch_sum / (60000//mb_size)
    d_loss_epoch_avg = d_loss_epoch_sum / (60000//mb_size)
    d_loss_real_epoch_avg = d_loss_real_epoch_sum / (60000//mb_size)
    d_loss_fake_epoch_avg = d_loss_fake_epoch_sum / (60000//mb_size)
    
    experiment.log_metrics({
        "D_loss_epoch": d_loss_epoch_avg,
        "G_loss_epoch": g_loss_epoch_avg,
        "D_loss_real": d_loss_real_epoch_avg,
        "D_loss_fake": d_loss_fake_epoch_avg
    }, step=epoch)
            
    #save image each epoch
    with torch.no_grad():
        z = torch.randn(mb_size, Z_dim).to(device)
        samples = G(z).detach().cpu()
    
        samples = samples.view(mb_size, 1, 28, 28)
    
        # make a proper 8x8 grid (64 images)
        grid = torch.zeros(28 * 8, 28 * 8)
    
        k = 0
        for i in range(8):
            for j in range(8):
                grid[i*28:(i+1)*28, j*28:(j+1)*28] = samples[k, 0]
                k += 1
    
        plt.imshow(grid, cmap='gray')
        plt.axis('off')
        plt.savefig(f"./task1&task2_images/task2_epoch_{epoch}.png")
        plt.close()
# Save model after last epoch
torch.save({
    'G_params': {
        'Wzh': Wzh.data,
        'bzh': bzh.data,
        'Whx': Whx.data,
        'bhx': bhx.data
    },
    'D_params': {
        'Wxh': Wxh.data,
        'bxh': bxh.data,
        'Why': Why.data,
        'bhy': bhy.data
    }
}, f"./models/task2_model_{run_name}.pth")
experiment.end()

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/drmrai/d7047e-lab2/9dc2e62df6e4469aa485cf3f30d8c949

COMET INFO: Couldn't find a Git repository in '/home/jovyan/lab2' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


Epoch 0
Iter 0 | D_loss: 1.5063 | G_loss: 2.5839
Iter 100 | D_loss: 0.4305 | G_loss: 3.3890
Iter 200 | D_loss: 0.0566 | G_loss: 4.9769
Iter 300 | D_loss: 0.0564 | G_loss: 5.1075
Iter 400 | D_loss: 0.0164 | G_loss: 5.8177
Iter 500 | D_loss: 0.0220 | G_loss: 5.7413
Iter 600 | D_loss: 0.0118 | G_loss: 6.2510
Iter 700 | D_loss: 0.0042 | G_loss: 7.4588
Iter 800 | D_loss: 0.0030 | G_loss: 7.9074
Iter 900 | D_loss: 0.0099 | G_loss: 8.2530
Epoch 1
Iter 0 | D_loss: 0.0068 | G_loss: 8.4505
Iter 100 | D_loss: 0.0177 | G_loss: 8.0532
Iter 200 | D_loss: 0.0034 | G_loss: 8.8616
Iter 300 | D_loss: 0.0023 | G_loss: 8.9253
Iter 400 | D_loss: 0.0022 | G_loss: 9.1632
Iter 500 | D_loss: 0.0051 | G_loss: 7.1631
Iter 600 | D_loss: 0.0134 | G_loss: 7.0989
Iter 700 | D_loss: 0.0052 | G_loss: 6.9374
Iter 800 | D_loss: 0.0120 | G_loss: 6.7503
Iter 900 | D_loss: 0.0096 | G_loss: 9.4442
Epoch 2
Iter 0 | D_loss: 0.0232 | G_loss: 7.3952
Iter 100 | D_loss: 0.0233 | G_loss: 8.1785
Iter 200 | D_loss: 0.0022 | G_loss: 